# Batch 04: velocity, displacement and time intervals
Read-only recomputation across the development pool. All nine reviewed fields remain unchanged. First-row derivatives remain undefined. Speed thresholds identify review candidates, not proven errors.
No neighbour-based replacement, outlier deletion, time-gap filling, fitted model or final-test access. Run from work_place or a child directory using the project environment.

In [1]:
from pathlib import Path
import json,hashlib
import numpy as np
import pyarrow.parquet as pq
from pyproj import Geod
from data3001.dataset import development_ids,track_path
root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'configs/dataset_manifest.json').exists())
manifest=json.loads((root/'configs/dataset_manifest.json').read_text())
assert hashlib.sha256((root/'docs/contract/data_contract_v1.0.xlsx').read_bytes()).hexdigest()==manifest['contract_sha256']
fields='ve vn velocity_ok gap net_displacement_km interval_net_ve interval_net_vn elapsed_seconds_from_previous absent_hours_before'.split()
geo=Geod(ellps='WGS84');nulls={k:0 for k in fields}
checks={k:0 for k in ['nonfinite_native_velocity','velocity_ok_mismatch','negative_or_nonfinite_source_gap','derived_null_not_exactly_first_row','elapsed_mismatch','absent_hours_mismatch','displacement_mismatch','interval_ve_mismatch','interval_vn_mismatch']}
counts={k:0 for k in ['IDs','rows','R_rows','R_first_rows','velocity_ok_false','gap_intervals','absent_hours','zero_displacement_rows','zero_displacement_R_rows']}
screens={scope:{kind:{str(th):0 for th in [3,5,10]} for kind in ['official_speed','interval_speed','interval_speed_contiguous_only']} for scope in ['development_all','R_current_observation']}
# Thresholds only describe candidates for review; none is an approved exclusion rule.
top={scope:{kind:[] for kind in ['official_speed','interval_speed']} for scope in screens}
max_gap=None;max_distance=None;max_deviation={k:0.0 for k in ['distance_km','ve_m_s','vn_m_s']};nonfirst_nulls={k:0 for k in ['net_displacement_km','interval_net_ve','interval_net_vn','elapsed_seconds_from_previous']}
def case(ident,a,j,speed):
    out={'id':ident,'source_obs':int(a['source_obs'][j]),'UTC':str(np.datetime64(int(a['time'][j]),'s')),'lat':float(a['lat'][j]),'lon':float(a['lon'][j]),'official_ve':float(a['ve'][j]),'official_vn':float(a['vn'][j]),'official_speed_m_s':float(np.hypot(a['ve'][j],a['vn'][j])),'screen_speed_m_s':float(speed),'interval_net_speed_m_s':None if j==0 else float(np.hypot(a['interval_net_ve'][j],a['interval_net_vn'][j])),'elapsed_seconds':None if j==0 else float(a['elapsed_seconds_from_previous'][j]),'native_gap_seconds':float(a['gap'][j]),'in_R':bool(a['in_R'][j]),'mapping_ok':bool(a['mapping_ok'][j])}
    if j>0:out.update({'previous_UTC':str(np.datetime64(int(a['time'][j-1]),'s')),'previous_lat':float(a['lat'][j-1]),'previous_lon':float(a['lon'][j-1])})
    return out
for ident in development_ids():
    tab=pq.read_table(track_path(ident),columns=fields+['time','lat','lon','lon_normalized','in_R','mapping_ok','source_obs'])
    a={k:tab[k].to_numpy() for k in tab.column_names};n=tab.num_rows;assert n==manifest['records'][str(ident)]['rows']
    for k in fields:nulls[k]+=tab[k].null_count
    counts['IDs']+=1;counts['rows']+=n;counts['R_rows']+=int(a['in_R'].sum());counts['R_first_rows']+=int(a['in_R'][0]);counts['velocity_ok_false']+=int((~a['velocity_ok']).sum())
    native_ok=np.isfinite(a['ve'])&np.isfinite(a['vn'])
    checks['nonfinite_native_velocity']+=int((~native_ok).sum());checks['velocity_ok_mismatch']+=int((a['velocity_ok']!=native_ok).sum());checks['negative_or_nonfinite_source_gap']+=int((~np.isfinite(a['gap'])|(a['gap']<0)).sum())
    t=a['time'];dt=np.diff(t)
    expected_dt=np.r_[np.nan,dt]
    checks['elapsed_mismatch']+=int((~np.isclose(a['elapsed_seconds_from_previous'],expected_dt,equal_nan=True,rtol=0,atol=0)).sum())
    absent=np.r_[-1,(dt/3600-1).astype(np.int64)]
    checks['absent_hours_mismatch']+=int((a['absent_hours_before']!=absent).sum());counts['gap_intervals']+=int((absent>0).sum());counts['absent_hours']+=int(absent.clip(0).sum())
    for k in nonfirst_nulls:
        missing=~np.isfinite(a[k]);expected=np.arange(n)==0
        checks['derived_null_not_exactly_first_row']+=int((missing!=expected).sum());nonfirst_nulls[k]+=int(missing[1:].sum())
    az,_,distance=geo.inv(a['lon_normalized'][:-1],a['lat'][:-1],a['lon_normalized'][1:],a['lat'][1:])
    expected_km=np.r_[np.nan,distance/1000];v=distance/dt
    expected_u=np.r_[np.nan,v*np.sin(np.deg2rad(az))];expected_v=np.r_[np.nan,v*np.cos(np.deg2rad(az))]
    for k,expected,check,key in [('net_displacement_km',expected_km,'displacement_mismatch','distance_km'),('interval_net_ve',expected_u,'interval_ve_mismatch','ve_m_s'),('interval_net_vn',expected_v,'interval_vn_mismatch','vn_m_s')]:
        checks[check]+=int((~np.isclose(a[k],expected,equal_nan=True,rtol=1e-12,atol=1e-9)).sum())
        if n>1:max_deviation[key]=max(max_deviation[key],float(np.nanmax(np.abs(a[k]-expected))))
    official=np.hypot(a['ve'],a['vn']);interval=np.hypot(a['interval_net_ve'],a['interval_net_vn']);contiguous=a['elapsed_seconds_from_previous']==3600
    counts['zero_displacement_rows']+=int((a['net_displacement_km']==0).sum());counts['zero_displacement_R_rows']+=int(((a['net_displacement_km']==0)&a['in_R']).sum())
    for scope,mask in [('development_all',np.ones(n,dtype=bool)),('R_current_observation',a['in_R'])]:
        for th in [3,5,10]:
            screens[scope]['official_speed'][str(th)]+=int((mask&(official>th)).sum());screens[scope]['interval_speed'][str(th)]+=int((mask&(interval>th)).sum());screens[scope]['interval_speed_contiguous_only'][str(th)]+=int((mask&contiguous&(interval>th)).sum())
        for kind,speed in [('official_speed',official),('interval_speed',interval)]:
            idx=np.flatnonzero(mask&np.isfinite(speed))
            if idx.size:
                picked=idx[np.argsort(speed[idx])[-3:]]
                top[scope][kind].extend(case(ident,a,int(j),speed[j]) for j in picked)
                top[scope][kind]=sorted(top[scope][kind],key=lambda x:x['screen_speed_m_s'],reverse=True)[:5]
    if n>1:
        j=int(np.argmax(a['elapsed_seconds_from_previous'][1:]))+1
        if max_gap is None or a['elapsed_seconds_from_previous'][j]>max_gap['elapsed_seconds']:
            max_gap=case(ident,a,j,interval[j]);max_gap['absent_hours_before']=int(absent[j]);max_gap['distance_km']=float(a['net_displacement_km'][j])
        j=int(np.argmax(a['net_displacement_km'][1:]))+1
        if max_distance is None or a['net_displacement_km'][j]>max_distance['distance_km']:
            max_distance=case(ident,a,j,interval[j]);max_distance['distance_km']=float(a['net_displacement_km'][j])
assert not any(checks.values()) and counts['rows']==33943292 and counts['IDs']==3224
for k in nonfirst_nulls:assert nulls[k]==3224 and nonfirst_nulls[k]==0
result={'scope':'Development only; final test not opened. Speed thresholds are descriptive screens, not deletion rules. R interval screens refer to the current endpoint being in R; the previous endpoint may be outside.','release':manifest['version'],'contract_sha256':manifest['contract_sha256'],'counts':counts,'null_rows_by_field':nulls,'nonfirst_null_rows':nonfirst_nulls,'consistency_failures':checks,'max_recalculation_difference':max_deviation,'speed_screens_m_s':screens,'top_speed_examples':top,'longest_missing_interval':max_gap,'largest_geodesic_displacement':max_distance,'input_file_checksums_verified':counts['IDs'],'data_modified':False,'imputation_performed':False,'outliers_adjudicated':False}
print(json.dumps(result,indent=2))


{
  "scope": "Development only; final test not opened. Speed thresholds are descriptive screens, not deletion rules. R interval screens refer to the current endpoint being in R; the previous endpoint may be outside.",
  "release": "per-drifter-20261016-v1",
  "contract_sha256": "6cfa91471a1399a9c3a1a8edfeb2e18cea17e0f4031ef0f8ca604ca4565a2446",
  "counts": {
    "IDs": 3224,
    "rows": 33943292,
    "R_rows": 318108,
    "R_first_rows": 56,
    "velocity_ok_false": 0,
    "gap_intervals": 11257,
    "absent_hours": 620388,
    "zero_displacement_rows": 988,
    "zero_displacement_R_rows": 1
  },
  "null_rows_by_field": {
    "ve": 0,
    "vn": 0,
    "velocity_ok": 0,
    "gap": 0,
    "net_displacement_km": 3224,
    "interval_net_ve": 3224,
    "interval_net_vn": 3224,
    "elapsed_seconds_from_previous": 3224,
    "absent_hours_before": 0
  },
  "nonfirst_null_rows": {
    "net_displacement_km": 0,
    "interval_net_ve": 0,
    "interval_net_vn": 0,
    "elapsed_seconds_from_previo

In [2]:
from data3001.dataset import track_path
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
ident=300234065382690
columns=['source_obs','time_UTC','lat','lon','ve','vn','net_displacement_km','interval_net_ve','interval_net_vn','elapsed_seconds_from_previous','gap','err_lat','err_lon','err_ve','err_vn','mapping_ok','cell_1','cell_05']
a=pq.read_table(track_path(ident),columns=columns).to_pandas()
t=pd.Timestamp('2020-08-17T10:00Z')
near=a[(a.time_UTC>=t-pd.Timedelta(hours=2))&(a.time_UTC<=t+pd.Timedelta(hours=2))].copy()
near['official_speed']=np.hypot(near.ve,near.vn)
near['interval_speed']=np.hypot(near.interval_net_ve,near.interval_net_vn)
print(near.to_string(index=False))


 source_obs                  time_UTC       lat        lon       ve        vn  net_displacement_km  interval_net_ve  interval_net_vn  elapsed_seconds_from_previous    gap      err_lat      err_lon   err_ve   err_vn  mapping_ok  cell_1  cell_05  official_speed  interval_speed
  158746555 2020-08-17 08:00:00+00:00 32.063474 -74.870825 0.734629 -0.029449           196.711273        -0.035720         0.235923                       824400.0 3628.0 8.634582e-10 1.019609e-09 0.000051 0.000051        True   44025   175890        0.735219        0.238611
  158746556 2020-08-17 09:00:00+00:00 32.057504 -74.834950 0.727671 -0.024084             3.451840         0.941077        -0.183732                         3600.0    0.0 8.505676e-10 1.004853e-09 0.000051 0.000051        True   44025   175890        0.728070        0.958845
  158746557 2020-08-17 10:00:00+00:00 32.608515 -74.894919 0.246595 -0.971550            61.363355        -1.563628        16.973507                         3600.0 3628.0 8